<a href="https://colab.research.google.com/github/elsheppardo/hello-world/blob/main/ACB_for_CAD_to_USD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
STAGE 2: Fiat Cost Basis — CAD→USD Purchases
==============================================
Purpose: Establish the CAD cost basis of the USD that was wired to
         Paxos in 2022. This produces a single number — the
         weighted-average CAD cost per $1 USD — that drives every
         USD→CAD disposition FX gain/loss in 2023.

Output: Stage2_Fiat_Cost_Basis.xlsx

Inputs (from user records):
  - Jan 3, 2022:  Bought $210,000 USD at FX rate 1.27
  - Mar 9, 2022:  Bought $76,845 USD at FX rate 1.28

Key output:
  - Total USD acquired: $286,845
  - Total CAD spent:    $365,061.60
  - Weighted avg rate (ACB per USD):  $1.27268 CAD per $1 USD
    (This is the number every Stage 5 USD→CAD disposition will use.)

Wire reconciliation (for sanity check):
  - Paxos wires totalled $286,815 USD (Mar 4: $114,970 + Mar 9: $171,845)
  - USD purchased totalled $286,845
  - Difference: $30 — small wire fee, immaterial

How to verify:
  1. Open the file.
  2. Confirm the two CAD→USD purchase rows (yellow cells) match
     your bank records: amounts and FX rates.
  3. The ACB per USD cell (B9) should compute to ~1.2727.
  4. If your actual FX rates differ (e.g., 1.2715 instead of 1.27),
     edit the yellow cells and the ACB per USD will recalculate.
"""

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.comments import Comment

FONT_NAME = "Arial"
styles = {
    'title':       Font(name=FONT_NAME, size=16, bold=True, color='1F4E79'),
    'section':     Font(name=FONT_NAME, size=12, bold=True, color='1F4E79'),
    'header':      Font(name=FONT_NAME, size=10, bold=True, color='FFFFFF'),
    'body':        Font(name=FONT_NAME, size=10),
    'body_bold':   Font(name=FONT_NAME, size=10, bold=True),
    'input':       Font(name=FONT_NAME, size=10, color='0000FF'),
    'formula':     Font(name=FONT_NAME, size=10, color='000000'),
    'key_output':  Font(name=FONT_NAME, size=12, bold=True, color='006100'),
    'note':        Font(name=FONT_NAME, size=9,  italic=True, color='595959'),
    'header_fill': PatternFill('solid', fgColor='1F4E79'),
    'verify_fill': PatternFill('solid', fgColor='FFF2CC'),
    'total_fill':  PatternFill('solid', fgColor='E2EFDA'),
    'key_fill':    PatternFill('solid', fgColor='C6EFCE'),
}
thin = Side(border_style='thin', color='BFBFBF')
cell_border = Border(left=thin, right=thin, top=thin, bottom=thin)

FMT_USD    = '"$"#,##0.00;("$"#,##0.00);"-"'
FMT_CAD    = '"$"#,##0.00;("$"#,##0.00);"-"'
FMT_FX     = '0.00000'


def build_workbook(output_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Fiat_Cost_Basis"

    # Column widths
    widths = [14, 16, 18, 16, 16, 16, 50]
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w

    # --- Title block ---
    ws.cell(row=1, column=1, value="Fiat Cost Basis — CAD→USD Purchases").font = styles['title']
    ws.cell(row=2, column=1, value=(
        "Establishes the CAD cost basis of the USD wired to Paxos in 2022. "
        "The ACB per USD at cell B9 is the key output referenced by later stages."
    )).font = styles['note']

    # --- Header row (row 4) ---
    headers = ["Date", "USD Bought", "FX Rate (CAD/USD)",
               "CAD Spent", "Cumulative USD", "Cumulative CAD", "Notes"]
    for i, h in enumerate(headers, 1):
        cell = ws.cell(row=4, column=i, value=h)
        cell.font = styles['header']
        cell.fill = styles['header_fill']
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = cell_border

    # --- Row 5: Jan 3, 2022 purchase ---
    r = 5
    ws.cell(row=r, column=1, value="2022-01-03").font = styles['body']
    c = ws.cell(row=r, column=2, value=210000);   c.font = styles['input'];   c.fill = styles['verify_fill']; c.number_format = FMT_USD
    c = ws.cell(row=r, column=3, value=1.27);     c.font = styles['input'];   c.fill = styles['verify_fill']; c.number_format = FMT_FX
    c = ws.cell(row=r, column=4, value=f"=B{r}*C{r}");  c.font = styles['formula']; c.number_format = FMT_CAD
    c = ws.cell(row=r, column=5, value=f"=B{r}");        c.font = styles['formula']; c.number_format = FMT_USD
    c = ws.cell(row=r, column=6, value=f"=D{r}");        c.font = styles['formula']; c.number_format = FMT_CAD
    ws.cell(row=r, column=7, value="First fiat purchase (per user records)").font = styles['note']
    for col in range(1, 8):
        ws.cell(row=r, column=col).border = cell_border

    # --- Row 6: Mar 9, 2022 purchase ---
    r = 6
    ws.cell(row=r, column=1, value="2022-03-09").font = styles['body']
    c = ws.cell(row=r, column=2, value=76845);    c.font = styles['input'];   c.fill = styles['verify_fill']; c.number_format = FMT_USD
    c = ws.cell(row=r, column=3, value=1.28);     c.font = styles['input'];   c.fill = styles['verify_fill']; c.number_format = FMT_FX
    c = ws.cell(row=r, column=4, value=f"=B{r}*C{r}");      c.font = styles['formula']; c.number_format = FMT_CAD
    c = ws.cell(row=r, column=5, value=f"=E{r-1}+B{r}");    c.font = styles['formula']; c.number_format = FMT_USD
    c = ws.cell(row=r, column=6, value=f"=F{r-1}+D{r}");    c.font = styles['formula']; c.number_format = FMT_CAD
    ws.cell(row=r, column=7, value="Top-up for Mar 9 wire (per user records)").font = styles['note']
    for col in range(1, 8):
        ws.cell(row=r, column=col).border = cell_border

    # --- Row 7: TOTAL ---
    r = 7
    c = ws.cell(row=r, column=1, value="TOTAL"); c.font = styles['body_bold']; c.fill = styles['total_fill']
    c = ws.cell(row=r, column=2, value="=SUM(B5:B6)");         c.font = styles['body_bold']; c.fill = styles['total_fill']; c.number_format = FMT_USD
    c = ws.cell(row=r, column=3, value=f"=D{r}/B{r}");          c.font = styles['body_bold']; c.fill = styles['total_fill']; c.number_format = FMT_FX
    c = ws.cell(row=r, column=4, value="=SUM(D5:D6)");         c.font = styles['body_bold']; c.fill = styles['total_fill']; c.number_format = FMT_CAD
    c = ws.cell(row=r, column=5, value="=E6");                  c.font = styles['body_bold']; c.fill = styles['total_fill']; c.number_format = FMT_USD
    c = ws.cell(row=r, column=6, value="=F6");                  c.font = styles['body_bold']; c.fill = styles['total_fill']; c.number_format = FMT_CAD
    ws.cell(row=r, column=7, value="Col C shows effective avg rate").font = styles['note']
    ws.cell(row=r, column=7).fill = styles['total_fill']
    for col in range(1, 8):
        ws.cell(row=r, column=col).border = cell_border

    # --- Row 9: KEY OUTPUT — ACB per USD ---
    r = 9
    c = ws.cell(row=r, column=1, value="ACB per USD (CAD):"); c.font = styles['key_output']; c.fill = styles['key_fill']
    c = ws.cell(row=r, column=2, value="=F7/E7")
    c.font = styles['key_output']; c.fill = styles['key_fill']; c.number_format = FMT_FX
    c.comment = Comment(
        "KEY OUTPUT: CAD cost basis per $1 USD. Weighted average of Jan 3 and Mar 9 purchases. "
        "Referenced by Stage 5 (2023 Events) to compute USD→CAD disposition gain/loss.",
        "Claude"
    )
    ws.cell(row=r, column=3, value="← This is the KEY OUTPUT referenced by later stages").font = styles['note']
    ws.cell(row=r, column=3).fill = styles['key_fill']

    # --- Wire Reconciliation section ---
    r = 12
    ws.cell(row=r, column=1, value="Wire Reconciliation (Sanity Check)").font = styles['section']
    r = 13
    headers2 = ["Date", "USD Wired to Paxos", "", "", "", "", "Notes"]
    for i, h in enumerate(headers2, 1):
        cell = ws.cell(row=r, column=i, value=h)
        if h:
            cell.font = styles['header']
            cell.fill = styles['header_fill']
            cell.alignment = Alignment(horizontal='center', vertical='center')
            cell.border = cell_border

    r = 14
    ws.cell(row=r, column=1, value="2022-03-04").font = styles['body']
    c = ws.cell(row=r, column=2, value=114970); c.font = styles['input']; c.fill = styles['verify_fill']; c.number_format = FMT_USD
    ws.cell(row=r, column=7, value="Paxos wire deposit #1 (from Paxos CSV)").font = styles['note']

    r = 15
    ws.cell(row=r, column=1, value="2022-03-09").font = styles['body']
    c = ws.cell(row=r, column=2, value=171845); c.font = styles['input']; c.fill = styles['verify_fill']; c.number_format = FMT_USD
    ws.cell(row=r, column=7, value="Paxos wire deposit #2 (from Paxos CSV)").font = styles['note']

    r = 16
    c = ws.cell(row=r, column=1, value="Total wired to Paxos"); c.font = styles['body_bold']
    c = ws.cell(row=r, column=2, value="=SUM(B14:B15)"); c.font = styles['body_bold']; c.number_format = FMT_USD

    r = 17
    c = ws.cell(row=r, column=1, value="USD purchased (from above)"); c.font = styles['body']
    c = ws.cell(row=r, column=2, value="=E7"); c.font = styles['formula']; c.number_format = FMT_USD

    r = 18
    c = ws.cell(row=r, column=1, value="Difference"); c.font = styles['body']
    c = ws.cell(row=r, column=2, value="=B16-B17"); c.font = styles['formula']; c.number_format = FMT_USD
    ws.cell(row=r, column=7, value="Small wire fee — immaterial").font = styles['note']

    # --- Instructions for downstream use ---
    r = 21
    ws.cell(row=r, column=1, value="How this sheet is used:").font = styles['section']
    instructions = [
        "",
        "Stage 5 (2023 Events) references this sheet's cell B9 (ACB per USD) to compute FX gain/loss",
        "on every USD→CAD conversion at NDAX in 2023.",
        "",
        "When you dispose of USD (by converting to CAD), the CAD gain/loss is:",
        "   CAD received  −  (USD disposed × ACB per USD from cell B9)",
        "",
        "If the yellow-highlighted FX rates (cells C5 and C6) are updated, the ACB per USD",
        "recalculates automatically, and all downstream calculations update.",
    ]
    for i, line in enumerate(instructions):
        c = ws.cell(row=r + i, column=1, value=line)
        c.font = styles['body'] if line else styles['note']

    ws.freeze_panes = "A5"
    wb.save(output_path)
    print(f"Saved: {output_path}")
    print()
    print("Expected key output (cell B9, ACB per USD):")
    print("  = (210000*1.27 + 76845*1.28) / (210000 + 76845)")
    print("  = (266700 + 98361.60) / 286845")
    print("  = 365061.60 / 286845")
    print(f"  = {365061.60/286845:.5f} CAD per $1 USD")


if __name__ == "__main__":
    build_workbook("Stage2_Fiat_Cost_Basis.xlsx")